In [1]:
import json
import math
import random
import time
from pathlib import Path

import torch
import pandas as pd
from transformers import AutoTokenizer, GPT2LMHeadModel

# ----------------------------
# Config
# ----------------------------
MODEL_DIR = Path("../models/models--gpt2").resolve()
PROMPT = "The secret code is 73914. Repeat the secret code exactly:"
SEED = 42

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

random.seed(SEED)
torch.manual_seed(SEED)
torch.set_grad_enabled(False)

print("device:", device)
print("model dir:", MODEL_DIR)
print("exists:", MODEL_DIR.exists())

device: mps
model dir: /Users/nelson/py/mechanistic_inter/models/models--gpt2
exists: True


In [2]:
assert MODEL_DIR.exists(), f"Missing model folder: {MODEL_DIR}"

print("Files in model dir:")
for f in sorted(MODEL_DIR.iterdir()):
    print(" ", f.name)

Files in model dir:
  config.json
  generation_config.json
  merges.txt
  model.safetensors
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json
  vocab.json


In [3]:
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

model = GPT2LMHeadModel.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

model = model.to(device=device, dtype=torch.float32).eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loaded model in {time.time() - t0:.2f}s")
print("n_layer:", model.config.n_layer)
print("n_head :", model.config.n_head)
print("n_embd :", model.config.n_embd)

Loaded model in 0.71s
n_layer: 12
n_head : 12
n_embd : 768


In [4]:
def decode_token(token_id: int) -> str:
    return repr(tokenizer.decode([int(token_id)]))

def token_strs(input_ids: torch.Tensor):
    return [repr(tokenizer.decode([int(x)])) for x in input_ids[0].tolist()]

def get_target_pos(input_ids: torch.Tensor) -> int:
    # the first generated token is predicted from the final prompt position
    return input_ids.shape[1] - 1

def get_last_logits(logits: torch.Tensor, pos: int) -> torch.Tensor:
    return logits[0, pos]

def chosen_and_foil(last_logits: torch.Tensor):
    chosen_id = int(last_logits.argmax().item())
    foil_logits = last_logits.clone()
    foil_logits[chosen_id] = -torch.inf
    foil_id = int(foil_logits.argmax().item())
    margin = float((last_logits[chosen_id] - last_logits[foil_id]).item())
    return chosen_id, foil_id, margin

def logit_diff_direction(lm_head_weight: torch.Tensor, chosen_id: int, foil_id: int):
    # lm_head.weight shape: [vocab, d_model]
    return lm_head_weight[chosen_id] - lm_head_weight[foil_id]

def parse_head_label(label: str):
    import re
    m = re.fullmatch(r"L(\d+)H(\d+)", label)
    if m is None:
        return None
    return int(m.group(1)), int(m.group(2))

def make_component_df(labels, contribs, top_k=20):
    df = pd.DataFrame({
        "label": labels,
        "contrib_to_margin": contribs.detach().cpu().float().numpy()
    })
    df["abs_contrib"] = df["contrib_to_margin"].abs()
    return df.sort_values("abs_contrib", ascending=False).head(top_k).reset_index(drop=True)

def make_signed_component_views(labels, contribs, top_k=20):
    df = pd.DataFrame({
        "label": labels,
        "contrib_to_margin": contribs.detach().cpu().float().numpy()
    })
    df["abs_contrib"] = df["contrib_to_margin"].abs()

    top_abs_df = df.sort_values("abs_contrib", ascending=False).head(top_k).reset_index(drop=True)
    top_supporters_df = (
        df[df["contrib_to_margin"] > 0]
        .sort_values("contrib_to_margin", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    top_opponents_df = (
        df[df["contrib_to_margin"] < 0]
        .sort_values("contrib_to_margin", ascending=True)
        .head(top_k)
        .reset_index(drop=True)
    )

    return {
        "all": df,
        "top_abs": top_abs_df,
        "top_supporters": top_supporters_df,
        "top_opponents": top_opponents_df,
    }

def top_token_candidates(last_logits: torch.Tensor, top_k=10):
    probs = torch.softmax(last_logits, dim=-1)
    top_logits, top_ids = torch.topk(last_logits, k=top_k)

    rows = []
    for rank, (token_id, logit) in enumerate(zip(top_ids.tolist(), top_logits.tolist()), start=1):
        token_id = int(token_id)
        rows.append({
            "rank": rank,
            "token_id": token_id,
            "token_str": decode_token(token_id),
            "logit": float(logit),
            "prob": float(probs[token_id].item()),
        })

    return pd.DataFrame(rows)

def linearized_ln_apply(delta: torch.Tensor, final_resid_pre_ln: torch.Tensor, ln_f_module):
    """
    Approximate final LayerNorm on a component delta using the statistics
    of the full final residual stream.
    """
    mean = final_resid_pre_ln.mean(dim=-1, keepdim=True)
    var = ((final_resid_pre_ln - mean) ** 2).mean(dim=-1, keepdim=True)
    std = torch.sqrt(var + ln_f_module.eps)

    centered_delta = delta - delta.mean(dim=-1, keepdim=True)
    gamma = ln_f_module.weight
    return centered_delta * (gamma / std)

def contrib_to_margin(delta: torch.Tensor, final_resid_pre_ln: torch.Tensor, ln_f_module, d: torch.Tensor):
    ln_delta = linearized_ln_apply(delta, final_resid_pre_ln, ln_f_module)
    return torch.einsum("...d,d->...", ln_delta, d)

In [5]:
def split_heads(x: torch.Tensor, n_head: int, head_dim: int):
    # [B, T, D] -> [B, H, T, Dh]
    B, T, D = x.shape
    return x.view(B, T, n_head, head_dim).permute(0, 2, 1, 3)

def recompute_gpt2_attention(attn_module, x_ln1: torch.Tensor):
    """
    Recompute GPT-2 attention internals from ln_1 output.
    Returns:
      attn_probs    [B, H, T, T]
      v             [B, H, T, Dh]
      per_head_proj [B, H, T, D]
      full_attn_out [B, T, D]
    """
    B, T, D = x_ln1.shape
    n_head = attn_module.num_heads
    head_dim = attn_module.head_dim

    qkv = attn_module.c_attn(x_ln1)  # [B, T, 3D]
    q, k, v = qkv.split(attn_module.split_size, dim=2)

    q = split_heads(q, n_head, head_dim)
    k = split_heads(k, n_head, head_dim)
    v = split_heads(v, n_head, head_dim)

    att = torch.matmul(q, k.transpose(-1, -2))  # [B, H, T, T]

    if getattr(attn_module, "scale_attn_weights", True):
        att = att / math.sqrt(head_dim)

    if getattr(attn_module, "scale_attn_by_inverse_layer_idx", False):
        if getattr(attn_module, "layer_idx", None) is not None:
            att = att / float(attn_module.layer_idx + 1)

    causal_mask = attn_module.bias[:, :, :T, :T].to(torch.bool)
    mask_value = torch.finfo(att.dtype).min
    att = torch.where(causal_mask, att, mask_value)

    attn_probs = torch.softmax(att, dim=-1)  # [B, H, T, T]
    context = torch.matmul(attn_probs, v)    # [B, H, T, Dh]

    # GPT-2 c_proj is Conv1D with weight [Dh_total, D]
    W = attn_module.c_proj.weight  # [D, D]
    b = attn_module.c_proj.bias    # [D]

    per_head_proj = []
    for h in range(n_head):
        W_h = W[h * head_dim:(h + 1) * head_dim, :]   # [Dh, D]
        out_h = context[:, h, :, :] @ W_h             # [B, T, D]
        per_head_proj.append(out_h)

    per_head_proj = torch.stack(per_head_proj, dim=1)  # [B, H, T, D]
    full_attn_out = per_head_proj.sum(dim=1) + b       # [B, T, D]

    return attn_probs, v, per_head_proj, full_attn_out

In [6]:
def attach_capture_hooks(model):
    tracked = {
        "ln1_out": {},
        "attn_out": {},
        "mlp_out": {},
        "final_resid_pre_ln": None,
    }
    handles = []

    for layer in range(model.config.n_layer):
        block = model.transformer.h[layer]

        def make_ln1_hook(layer_idx):
            def hook(module, inputs, output):
                tracked["ln1_out"][layer_idx] = output.detach()
            return hook

        def make_attn_hook(layer_idx):
            def hook(module, inputs, output):
                attn_output = output[0] if isinstance(output, tuple) else output
                tracked["attn_out"][layer_idx] = attn_output.detach()
            return hook

        def make_mlp_hook(layer_idx):
            def hook(module, inputs, output):
                tracked["mlp_out"][layer_idx] = output.detach()
            return hook

        handles.append(block.ln_1.register_forward_hook(make_ln1_hook(layer)))
        handles.append(block.attn.register_forward_hook(make_attn_hook(layer)))
        handles.append(block.mlp.register_forward_hook(make_mlp_hook(layer)))

    def ln_f_prehook(module, inputs):
        tracked["final_resid_pre_ln"] = inputs[0].detach()

    handles.append(model.transformer.ln_f.register_forward_pre_hook(ln_f_prehook))

    return tracked, handles

def remove_handles(handles):
    for h in handles:
        h.remove()

In [7]:
tracked, handles = attach_capture_hooks(model)

inputs = tokenizer(PROMPT, return_tensors="pt")
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

t0 = time.time()
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True,
    )
print(f"Forward pass in {time.time() - t0:.2f}s")

remove_handles(handles)

target_pos = get_target_pos(input_ids)
last_logits = get_last_logits(outputs.logits, target_pos)
chosen_id, foil_id, margin = chosen_and_foil(last_logits)

W_U = model.lm_head.weight.detach()
d = logit_diff_direction(W_U, chosen_id, foil_id)
candidate_df = top_token_candidates(last_logits, top_k=10)

print("Prompt tokens:")
print(token_strs(input_ids))
print()
print("Decision target position:", target_pos)
print("Last prompt token      :", decode_token(int(input_ids[0, target_pos].item())))
print("Notebook question      : what token wins at the next step?")
print()
print("Chosen token:", chosen_id, decode_token(chosen_id))
print("Foil token  :", foil_id, decode_token(foil_id))
print("Margin      :", round(margin, 4))
print()
print("Top next-token candidates:")
display(candidate_df)

Forward pass in 0.20s
Prompt tokens:
["'The'", "' secret'", "' code'", "' is'", "' 7'", "'39'", "'14'", "'.'", "' Repeat'", "' the'", "' secret'", "' code'", "' exactly'", "':'"]

Decision target position: 13
Last prompt token      : ':'
Notebook question      : what token wins at the next step?

Chosen token: 198 '\n'
Foil token  : 767 ' 7'
Margin      : 0.6601

Top next-token candidates:


,rank,token_id,token_str,logit,prob
0,1,198,'\n',-106.427391,0.181237
1,2,767,' 7',-107.087532,0.093659
2,3,860,' 9',-108.571686,0.021232
3,4,262,' the',-108.607323,0.020489
4,5,718,' 6',-108.608009,0.020475
5,6,352,' 1',-108.814331,0.016658
6,7,807,' 8',-108.830124,0.016397
7,8,366,"' ""'",-108.939812,0.014693
8,9,340,' it',-109.004494,0.013773
9,10,628,'\n\n',-109.012573,0.013662


In [8]:
final_resid_pre_ln = tracked["final_resid_pre_ln"][0, target_pos]  # [D]

labels = []
deltas = []

# embedding stream
embed_delta = outputs.hidden_states[0][0, target_pos].detach()
labels.append("embed")
deltas.append(embed_delta)

# each block's attn and mlp write
for layer in range(model.config.n_layer):
    labels.append(f"attn_L{layer}")
    deltas.append(tracked["attn_out"][layer][0, target_pos])

    labels.append(f"mlp_L{layer}")
    deltas.append(tracked["mlp_out"][layer][0, target_pos])

delta_stack = torch.stack(deltas, dim=0)  # [components, D]

component_contribs = contrib_to_margin(
    delta_stack,
    final_resid_pre_ln,
    model.transformer.ln_f,
    d,
)

component_views = make_signed_component_views(labels, component_contribs, top_k=30)
top_components_df = component_views["top_abs"]
top_supporter_components_df = component_views["top_supporters"]
top_opponent_components_df = component_views["top_opponents"]

print("Top supporters of the chosen-over-foil margin:")
display(top_supporter_components_df.head(10))
print()
print("Top opponents of the chosen-over-foil margin:")
display(top_opponent_components_df.head(10))
print()
print("Largest components by absolute effect:")
display(top_components_df.head(10))

Top supporters of the chosen-over-foil margin:


,label,contrib_to_margin,abs_contrib
0,mlp_L10,1.056128,1.056128
1,mlp_L11,0.489845,0.489845
2,mlp_L6,0.280119,0.280119
3,mlp_L8,0.232805,0.232805
4,attn_L0,0.182271,0.182271
5,mlp_L0,0.166617,0.166617
6,mlp_L1,0.131692,0.131692
7,mlp_L5,0.126060,0.126060
8,mlp_L3,0.070097,0.070097
9,attn_L2,0.042830,0.042830



Top opponents of the chosen-over-foil margin:


,label,contrib_to_margin,abs_contrib
0,attn_L9,-1.606357,1.606357
1,attn_L10,-1.316812,1.316812
2,attn_L11,-0.929835,0.929835
3,attn_L8,-0.817371,0.817371
4,attn_L7,-0.289585,0.289585
5,attn_L6,-0.173310,0.173310
6,attn_L1,-0.132721,0.132721
7,attn_L4,-0.049231,0.049231
8,attn_L5,-0.005921,0.005921



Largest components by absolute effect:


,label,contrib_to_margin,abs_contrib
0,attn_L9,-1.606357,1.606357
1,attn_L10,-1.316812,1.316812
2,mlp_L10,1.056128,1.056128
3,attn_L11,-0.929835,0.929835
4,attn_L8,-0.817371,0.817371
5,mlp_L11,0.489845,0.489845
6,attn_L7,-0.289585,0.289585
7,mlp_L6,0.280119,0.280119
8,mlp_L8,0.232805,0.232805
9,attn_L0,0.182271,0.182271


In [9]:
head_labels = []
head_deltas = []

per_layer_head_outputs = {}
per_layer_attn_probs = {}
per_layer_v = {}

for layer in range(model.config.n_layer):
    x_ln1 = tracked["ln1_out"][layer]  # [B, T, D]
    attn_module = model.transformer.h[layer].attn

    attn_probs, v, per_head_proj, full_attn_out = recompute_gpt2_attention(attn_module, x_ln1)

    per_layer_head_outputs[layer] = per_head_proj.detach()
    per_layer_attn_probs[layer] = attn_probs.detach()
    per_layer_v[layer] = v.detach()

    for head in range(model.config.n_head):
        head_labels.append(f"L{layer}H{head}")
        head_deltas.append(per_head_proj[0, head, target_pos])

head_delta_stack = torch.stack(head_deltas, dim=0)  # [num_heads_total, D]

head_contribs = contrib_to_margin(
    head_delta_stack,
    final_resid_pre_ln,
    model.transformer.ln_f,
    d,
)

head_views = make_signed_component_views(head_labels, head_contribs, top_k=20)
top_heads_df = head_views["top_abs"]
top_supporter_heads_df = head_views["top_supporters"]
top_opponent_heads_df = head_views["top_opponents"]

print("Top head supporters of the chosen-over-foil margin:")
display(top_supporter_heads_df.head(10))
print()
print("Top head opponents of the chosen-over-foil margin:")
display(top_opponent_heads_df.head(10))
print()
print("Largest heads by absolute effect:")
display(top_heads_df.head(10))

Top head supporters of the chosen-over-foil margin:


,label,contrib_to_margin,abs_contrib
0,L10H7,0.444999,0.444999
1,L9H5,0.216259,0.216259
2,L8H10,0.200157,0.200157
3,L7H6,0.124473,0.124473
4,L0H1,0.105810,0.105810
5,L2H3,0.102605,0.102605
6,L8H2,0.094698,0.094698
7,L6H3,0.072965,0.072965
8,L3H6,0.069293,0.069293
9,L2H0,0.067413,0.067413



Top head opponents of the chosen-over-foil margin:


,label,contrib_to_margin,abs_contrib
0,L9H9,-1.504223,1.504223
1,L10H2,-1.351050,1.351050
2,L11H4,-0.427137,0.427137
3,L11H0,-0.319134,0.319134
4,L8H11,-0.294564,0.294564
5,L8H3,-0.276338,0.276338
6,L8H8,-0.250774,0.250774
7,L8H0,-0.225638,0.225638
8,L10H9,-0.169354,0.169354
9,L11H3,-0.159240,0.159240



Largest heads by absolute effect:


,label,contrib_to_margin,abs_contrib
0,L9H9,-1.504223,1.504223
1,L10H2,-1.351050,1.351050
2,L10H7,0.444999,0.444999
3,L11H4,-0.427137,0.427137
4,L11H0,-0.319134,0.319134
5,L8H11,-0.294564,0.294564
6,L8H3,-0.276338,0.276338
7,L8H8,-0.250774,0.250774
8,L8H0,-0.225638,0.225638
9,L9H5,0.216259,0.216259


In [10]:
def trace_head_sources(layer, head, target_pos, final_resid_pre_ln, d, top_k=12):
    attn_probs = per_layer_attn_probs[layer]  # [B, H, T, T]
    v = per_layer_v[layer]                    # [B, H, T, Dh]
    attn_module = model.transformer.h[layer].attn

    head_dim = attn_module.head_dim
    W = attn_module.c_proj.weight  # [D, D]
    W_h = W[head * head_dim:(head + 1) * head_dim, :]  # [Dh, D]

    probs = attn_probs[0, head, target_pos]  # [T]
    v_src = v[0, head]                       # [T, Dh]

    source_deltas = probs.unsqueeze(-1) * v_src   # [T, Dh]
    source_deltas = source_deltas @ W_h           # [T, D]

    source_contribs = contrib_to_margin(
        source_deltas,
        final_resid_pre_ln.unsqueeze(0),
        model.transformer.ln_f,
        d,
    )

    df = pd.DataFrame({
        "src_pos": list(range(input_ids.shape[1])),
        "src_token": token_strs(input_ids),
        "attention_weight": probs.detach().cpu().float().numpy(),
        "contrib_to_margin": source_contribs.detach().cpu().float().numpy(),
    })
    df["abs_contrib"] = df["contrib_to_margin"].abs()
    return df.sort_values("abs_contrib", ascending=False).head(top_k).reset_index(drop=True)

top_head_label = top_heads_df.iloc[0]["label"]
layer, head = parse_head_label(top_head_label)

print("Tracing head:", top_head_label)
source_df = trace_head_sources(layer, head, target_pos, final_resid_pre_ln, d, top_k=12)
source_df

Tracing head: L9H9


,src_pos,src_token,attention_weight,contrib_to_margin,abs_contrib
0,4,' 7',0.652214,-1.498321,1.498321
1,5,'39',0.065591,-0.017029,0.017029
2,8,' Repeat',0.007950,0.004104,0.004104
3,10,' secret',0.005902,0.003987,0.003987
4,0,'The',0.248624,0.003072,0.003072
5,7,'.',0.003075,-0.001538,0.001538
6,1,' secret',0.002390,0.001168,0.001168
7,2,' code',0.003518,0.000959,0.000959
8,6,'14',0.003799,-0.000658,0.000658
9,9,' the',0.001360,-0.000279,0.000279


In [11]:
top3_head_traces = {}

for label in top_heads_df.head(3)["label"]:
    parsed = parse_head_label(label)
    if parsed is None:
        continue
    l, h = parsed
    top3_head_traces[label] = trace_head_sources(
        l, h, target_pos, final_resid_pre_ln, d, top_k=8
    )

for label, df in top3_head_traces.items():
    print("=" * 80)
    print("HEAD:", label)
    display(df)

HEAD: L9H9


,src_pos,src_token,attention_weight,contrib_to_margin,abs_contrib
0,4,' 7',0.652214,-1.498321,1.498321
1,5,'39',0.065591,-0.017029,0.017029
2,8,' Repeat',0.007950,0.004104,0.004104
3,10,' secret',0.005902,0.003987,0.003987
4,0,'The',0.248624,0.003072,0.003072
5,7,'.',0.003075,-0.001538,0.001538
6,1,' secret',0.002390,0.001168,0.001168
7,2,' code',0.003518,0.000959,0.000959


HEAD: L10H2


,src_pos,src_token,attention_weight,contrib_to_margin,abs_contrib
0,4,' 7',0.206762,-1.233608,1.233608
1,5,'39',0.110868,-0.225832,0.225832
2,0,'The',0.544336,0.128107,0.128107
3,6,'14',0.036099,-0.070622,0.070622
4,13,':',0.036819,0.016739,0.016739
5,8,' Repeat',0.017348,0.009173,0.009173
6,10,' secret',0.004342,0.006520,0.006520
7,2,' code',0.008061,0.006006,0.006006


HEAD: L10H7


,src_pos,src_token,attention_weight,contrib_to_margin,abs_contrib
0,4,' 7',0.248619,0.499550,0.499550
1,0,'The',0.486306,-0.028149,0.028149
2,5,'39',0.021335,0.016184,0.016184
3,13,':',0.045975,-0.014406,0.014406
4,3,' is',0.023820,-0.012071,0.012071
5,9,' the',0.022341,-0.010978,0.010978
6,6,'14',0.020421,0.009860,0.009860
7,7,'.',0.021382,-0.008306,0.008306


In [12]:
def build_selected_head_removal_hook(layer_idx, selected_heads, target_pos):
    def hook(module, inputs, output):
        attn_output = output[0] if isinstance(output, tuple) else output
        x_ln1 = inputs[0]

        if len(selected_heads) == 0:
            return output

        _, _, per_head_proj, _ = recompute_gpt2_attention(module, x_ln1)
        remove_delta = per_head_proj[:, selected_heads, target_pos, :].sum(dim=1)  # [B, D]

        new_attn_output = attn_output.clone()
        new_attn_output[:, target_pos, :] = new_attn_output[:, target_pos, :] - remove_delta

        if isinstance(output, tuple):
            return (new_attn_output,) + output[1:]
        return new_attn_output

    return hook

def run_with_head_ablation(input_ids, attention_mask, heads_to_remove):
    layer_to_heads = {}
    for layer, head in heads_to_remove:
        layer_to_heads.setdefault(layer, []).append(head)

    handles = []
    for layer, selected_heads in layer_to_heads.items():
        module = model.transformer.h[layer].attn
        handles.append(
            module.register_forward_hook(
                build_selected_head_removal_hook(layer, selected_heads, target_pos)
            )
        )

    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

    for h in handles:
        h.remove()

    return out

top_head_pairs = []
for label in top_heads_df.head(3)["label"]:
    parsed = parse_head_label(label)
    if parsed is not None:
        top_head_pairs.append(parsed)

print("Ablating heads:", top_head_pairs)

ablated_outputs = run_with_head_ablation(input_ids, attention_mask, top_head_pairs)
ablated_last_logits = get_last_logits(ablated_outputs.logits, target_pos)
ablated_margin = float((ablated_last_logits[chosen_id] - ablated_last_logits[foil_id]).item())
ablated_top_id = int(ablated_last_logits.argmax().item())

necessity_score = (margin - ablated_margin) / max(1e-8, margin)

print("Original margin :", round(margin, 4))
print("Ablated margin  :", round(ablated_margin, 4))
print("Necessity score :", round(float(necessity_score), 4))
print("New top token   :", ablated_top_id, decode_token(ablated_top_id))
print("Kept original?  :", ablated_top_id == chosen_id)

Ablating heads: [(9, 9), (10, 2), (10, 7)]
Original margin : 0.6601
Ablated margin  : 2.4203
Necessity score : -2.6664
New top token   : 198 '\n'
Kept original?  : True


In [13]:
def build_keep_only_head_hook(layer_idx, keep_heads, target_pos):
    def hook(module, inputs, output):
        attn_output = output[0] if isinstance(output, tuple) else output
        x_ln1 = inputs[0]

        _, _, per_head_proj, _ = recompute_gpt2_attention(module, x_ln1)

        if len(keep_heads) == 0:
            keep_delta = torch.zeros_like(attn_output[:, target_pos, :])
        else:
            keep_delta = per_head_proj[:, keep_heads, target_pos, :].sum(dim=1)

        new_attn_output = attn_output.clone()
        new_attn_output[:, target_pos, :] = keep_delta

        if isinstance(output, tuple):
            return (new_attn_output,) + output[1:]
        return new_attn_output

    return hook

def run_keep_only_heads(input_ids, attention_mask, heads_to_keep):
    layer_to_heads = {}
    for layer, head in heads_to_keep:
        layer_to_heads.setdefault(layer, []).append(head)

    handles = []
    for layer in range(model.config.n_layer):
        keep_heads = layer_to_heads.get(layer, [])
        module = model.transformer.h[layer].attn
        handles.append(
            module.register_forward_hook(
                build_keep_only_head_hook(layer, keep_heads, target_pos)
            )
        )

    with torch.no_grad():
        out = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

    for h in handles:
        h.remove()

    return out

suff_outputs = run_keep_only_heads(input_ids, attention_mask, top_head_pairs)
suff_last_logits = get_last_logits(suff_outputs.logits, target_pos)
suff_margin = float((suff_last_logits[chosen_id] - suff_last_logits[foil_id]).item())
suff_top_id = int(suff_last_logits.argmax().item())

sufficiency_score = suff_margin / max(1e-8, margin)

print("Original margin   :", round(margin, 4))
print("Keep-only margin  :", round(suff_margin, 4))
print("Sufficiency score :", round(float(sufficiency_score), 4))
print("New top token     :", suff_top_id, decode_token(suff_top_id))
print("Kept original?    :", suff_top_id == chosen_id)

Original margin   : 0.6601
Keep-only margin  : 3.7091
Sufficiency score : 5.6187
New top token     : 198 '\n'
Kept original?    : True


In [14]:
all_heads = [
    (layer, head)
    for layer in range(model.config.n_layer)
    for head in range(model.config.n_head)
]

random_heads = random.sample(all_heads, k=len(top_head_pairs))

print("Chosen heads :", top_head_pairs)
print("Random heads :", random_heads)

rand_outputs = run_with_head_ablation(input_ids, attention_mask, random_heads)
rand_last_logits = get_last_logits(rand_outputs.logits, target_pos)
rand_margin = float((rand_last_logits[chosen_id] - rand_last_logits[foil_id]).item())

random_necessity_score = (margin - rand_margin) / max(1e-8, margin)

print("Top-head necessity score   :", round(float(necessity_score), 4))
print("Random-head necessity score:", round(float(random_necessity_score), 4))

Chosen heads : [(9, 9), (10, 2), (10, 7)]
Random heads : [(2, 4), (0, 6), (5, 10)]
Top-head necessity score   : -2.6664
Random-head necessity score: -0.1529


In [15]:
summary_df = pd.DataFrame([
    {"metric": "original_margin", "value": margin},
    {"metric": "necessity_margin_after_ablation", "value": ablated_margin},
    {"metric": "necessity_score", "value": float(necessity_score)},
    {"metric": "sufficiency_margin_keep_only", "value": suff_margin},
    {"metric": "sufficiency_score", "value": float(sufficiency_score)},
    {"metric": "random_necessity_score", "value": float(random_necessity_score)},
])

summary_df

,metric,value
0,original_margin,0.660141
1,necessity_margin_after_ablation,2.420334
2,necessity_score,-2.666389
3,sufficiency_margin_keep_only,3.709141
4,sufficiency_score,5.618710
5,random_necessity_score,-0.152879


In [16]:
def df_to_records(df):
    return json.loads(df.to_json(orient="records"))

report = {
    "meta": {
        "model_dir": str(MODEL_DIR),
        "prompt": PROMPT,
        "device": device,
        "target_pos": int(target_pos),
    },
    "decision": {
        "chosen_token_id": int(chosen_id),
        "chosen_token_str": decode_token(chosen_id),
        "foil_token_id": int(foil_id),
        "foil_token_str": decode_token(foil_id),
        "margin": float(margin),
    },
    "decision_top_candidates": df_to_records(candidate_df),
    "top_components": df_to_records(top_components_df),
    "top_component_supporters": df_to_records(top_supporter_components_df),
    "top_component_opponents": df_to_records(top_opponent_components_df),
    "top_heads": df_to_records(top_heads_df),
    "top_head_supporters": df_to_records(top_supporter_heads_df),
    "top_head_opponents": df_to_records(top_opponent_heads_df),
    "top_head_source_trace": {
        label: df_to_records(df) for label, df in top3_head_traces.items()
    },
    "tests": {
        "necessity": {
            "heads_ablated": [list(x) for x in top_head_pairs],
            "new_margin": float(ablated_margin),
            "score": float(necessity_score),
            "new_top_token_id": int(ablated_top_id),
            "new_top_token_str": decode_token(ablated_top_id),
            "kept_original_choice": bool(ablated_top_id == chosen_id),
        },
        "sufficiency": {
            "heads_kept": [list(x) for x in top_head_pairs],
            "new_margin": float(suff_margin),
            "score": float(sufficiency_score),
            "new_top_token_id": int(suff_top_id),
            "new_top_token_str": decode_token(suff_top_id),
            "kept_original_choice": bool(suff_top_id == chosen_id),
        },
        "random_control": {
            "heads_ablated": [list(x) for x in random_heads],
            "new_margin": float(rand_margin),
            "score": float(random_necessity_score),
        },
    },
}

report["decision"]

{'chosen_token_id': 198,
 'chosen_token_str': "'\\n'",
 'foil_token_id': 767,
 'foil_token_str': "' 7'",
 'margin': 0.6601409912109375}

In [17]:
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUTPUT_DIR / "gpt2_single_decision_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print("Saved:", out_path.resolve())

Saved: /Users/nelson/py/mechanistic_inter/notebook/outputs/gpt2_single_decision_report.json


In [18]:
print("PROMPT:")
print(PROMPT)
print()

print("CHOSEN:", decode_token(chosen_id))
print("FOIL  :", decode_token(foil_id))
print("MARGIN:", round(margin, 4))
print()

print("TOP NEXT-TOKEN CANDIDATES:")
display(candidate_df)
print()

print("TOP COMPONENT SUPPORTERS:")
display(top_supporter_components_df.head(5))
print()

print("TOP COMPONENT OPPONENTS:")
display(top_opponent_components_df.head(5))

print("TOP HEAD SUPPORTERS:")
display(top_supporter_heads_df.head(5))
print()

print("TOP HEAD OPPONENTS:")
display(top_opponent_heads_df.head(5))

print("NECESSITY SCORE :", round(float(necessity_score), 4))
print("SUFFICIENCY SCORE:", round(float(sufficiency_score), 4))
print("RANDOM CONTROL   :", round(float(random_necessity_score), 4))

PROMPT:
The secret code is 73914. Repeat the secret code exactly:

CHOSEN: '\n'
FOIL  : ' 7'
MARGIN: 0.6601

TOP NEXT-TOKEN CANDIDATES:


,rank,token_id,token_str,logit,prob
0,1,198,'\n',-106.427391,0.181237
1,2,767,' 7',-107.087532,0.093659
2,3,860,' 9',-108.571686,0.021232
3,4,262,' the',-108.607323,0.020489
4,5,718,' 6',-108.608009,0.020475
5,6,352,' 1',-108.814331,0.016658
6,7,807,' 8',-108.830124,0.016397
7,8,366,"' ""'",-108.939812,0.014693
8,9,340,' it',-109.004494,0.013773
9,10,628,'\n\n',-109.012573,0.013662



TOP COMPONENT SUPPORTERS:


,label,contrib_to_margin,abs_contrib
0,mlp_L10,1.056128,1.056128
1,mlp_L11,0.489845,0.489845
2,mlp_L6,0.280119,0.280119
3,mlp_L8,0.232805,0.232805
4,attn_L0,0.182271,0.182271



TOP COMPONENT OPPONENTS:


...

TOP HEAD SUPPORTERS:


,label,contrib_to_margin,abs_contrib
0,L10H7,0.444999,0.444999
1,L9H5,0.216259,0.216259
2,L8H10,0.200157,0.200157
3,L7H6,0.124473,0.124473
4,L0H1,0.105810,0.105810



TOP HEAD OPPONENTS:


,label,contrib_to_margin,abs_contrib
0,L9H9,-1.504223,1.504223
1,L10H2,-1.351050,1.351050
2,L11H4,-0.427137,0.427137
3,L11H0,-0.319134,0.319134
4,L8H11,-0.294564,0.294564


NECESSITY SCORE : -2.6664
SUFFICIENCY SCORE: 5.6187
RANDOM CONTROL   : -0.1529
